In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('d:/university works/Final-Year_Firts_sem/FYP/INFO/REPO/CA-Classification-Framework')

In [2]:
import pandas as pd

DATASET = PROJECT_ROOT / "datasets" / "context_augmented_dataset_preprocessed.jsonl"

working_df = pd.read_json(
    DATASET,
    lines=True
)

working_df.shape

(2745, 15)

In [3]:
sample = working_df.iloc[0].to_dict()

sample.keys()

dict_keys(['id', 'test_id', 'isFlaky', 'issue_category', 'repo_url', 'issue_commit', 'fixed_commit', 'test_code', 'helper_methods_json', 'failure_log', 'code_under_test_json', 'test_code_original', 'helper_methods_json_original', 'failure_log_original', 'code_under_test_original'])

In [4]:
from utils.prompt_builder import build_input_section

print(
    build_input_section(
        sample,
        include_context=False
    )[:3000]
)

## Input

### 1. Test Code

@Test
  public void testFindItemsByNamespace() throws Exception {
    final int page = 0;
    final int size = 50;
    final ArgumentCaptor<HttpGet> request = ArgumentCaptor.forClass(HttpGet.class);

    itemOpenApiService.findItemsByNamespace(someAppId, someEnv, someCluster, someNamespace, page, size);

    verify(httpClient, times(1)).execute(request.capture());

    HttpGet get = request.getValue();

    assertEquals(String.format("%s/envs/%s/apps/%s/clusters/%s/namespaces/%s/items?size=%s&page=%s",
            someBaseUrl, someEnv, someAppId, someCluster, someNamespace, size, page), get.getURI().toString());
  }


In [5]:
print(
    build_input_section(
        sample,
        include_context=True
    )[:6000]
)

## Input

### 1. Test Code

@Test
  public void testFindItemsByNamespace() throws Exception {
    final int page = 0;
    final int size = 50;
    final ArgumentCaptor<HttpGet> request = ArgumentCaptor.forClass(HttpGet.class);

    itemOpenApiService.findItemsByNamespace(someAppId, someEnv, someCluster, someNamespace, page, size);

    verify(httpClient, times(1)).execute(request.capture());

    HttpGet get = request.getValue();

    assertEquals(String.format("%s/envs/%s/apps/%s/clusters/%s/namespaces/%s/items?size=%s&page=%s",
            someBaseUrl, someEnv, someAppId, someCluster, someNamespace, size, page), get.getURI().toString());
  }

### 3. Failure Log

org.junit.ComparisonFailure: expected:<...someNamespace/items?[size=50&page=]0> but was:<...someNamespace/items?[page=0&size=5]0>
	at org.junit.Assert.assertEquals(Assert.java:117)
	at org.junit.Assert.assertEquals(Assert.java:146)
	at com.ctrip.framework.apollo.openapi.client.service.ItemOpenApiServiceTest.testFindItemsByNam

In [6]:
from utils.prompt_builder import build_prompt

prompt = build_prompt(
    sample=sample,
    strategy="zero_shot",
    include_context=True
)

print(prompt[:8000])

You are an expert software testing assistant specializing in flaky test identification and classification.

Base your analysis only on the provided artifacts. Do not make assumptions beyond the supplied information.

Return your answer strictly in the required JSON format.

Analyze the provided software testing artifacts and:

1. Classify the test as Flaky or Non-Flaky.
2. If the test is Flaky, identify the most appropriate flaky test category.
3. Explain your reasoning using only the available evidence.
4. Identify which artifacts support your conclusion.

Analyze the following test case and produce the final classification.

## Input

### 1. Test Code

@Test
  public void testFindItemsByNamespace() throws Exception {
    final int page = 0;
    final int size = 50;
    final ArgumentCaptor<HttpGet> request = ArgumentCaptor.forClass(HttpGet.class);

    itemOpenApiService.findItemsByNamespace(someAppId, someEnv, someCluster, someNamespace, page, size);

    verify(httpClient, times(1)